In [ ]:
# Step 0 --- check the workbench before we start.
# This cell only looks; it changes nothing. Run it and read the last line.

import sys
from pathlib import Path

print("Python  :", sys.version.split()[0])
print("Folder  :", Path.cwd().name)

_missing = []
for _name in ['numpy', 'pandas', 'sklearn', 'joblib']:
    try:
        __import__(_name)
    except ImportError:
        _missing.append(_name)

for _name in ['numpy', 'pandas', 'sklearn', 'joblib']:
    _mark = "missing" if _name in _missing else "ok"
    print(f"  {_name:<14} {_mark}")

if _missing:
    print()
    print("STOP. Some libraries are missing:", ", ".join(_missing))
    print("Ask your instructor to run the setup in labs/SETUP.md.")
else:
    print()
    print("All good. You can carry on to Step 1.")

Python  : 3.10.9
Folder  : P04-package
  numpy          ok
  pandas         ok
  sklearn        ok
  joblib         ok

All good. You can carry on to Step 1.


---

## Step 0b --- the dataset

Every practical in this course uses the same 600 food deliveries.
The next cell makes sure the file is there.

In [ ]:
# The delivery-time dataset every practical in this course uses.
# If the file is missing we build it again from the same seed, so every
# student in the room gets byte-for-byte the same 600 rows.

import csv
from pathlib import Path

import numpy as np

SEED = 42
N_ROWS = 600
DATA = Path("..") / "data" / "delivery_times.csv"


def make_delivery_csv(path=DATA):
    """Write the 600-row delivery dataset. Same formula as the lectures."""
    rng = np.random.default_rng(SEED)
    distance_km = np.round(rng.uniform(0.5, 12.0, N_ROWS), 2)
    prep_time_min = np.round(rng.uniform(5, 30, N_ROWS), 0)
    traffic_level = rng.integers(1, 4, N_ROWS)
    rain = rng.binomial(1, 0.25, N_ROWS)
    delivery_min = np.round(
        6.0
        + 3.1 * distance_km
        + 0.65 * prep_time_min
        + 4.2 * traffic_level
        + 5.5 * rain
        + rng.normal(0, 2.5, N_ROWS),
        1,
    )
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", newline="", encoding="utf-8") as fh:
        w = csv.writer(fh)
        w.writerow(["distance_km", "prep_time_min", "traffic_level",
                    "rain", "delivery_min"])
        for i in range(N_ROWS):
            w.writerow([distance_km[i], int(prep_time_min[i]),
                        int(traffic_level[i]), int(rain[i]), delivery_min[i]])
    return path


if not DATA.exists():
    make_delivery_csv()
    print("dataset rebuilt ->", DATA)
else:
    print("dataset found   ->", DATA)

dataset found   -> ..\data\delivery_times.csv


### Step 1 --- See the problem before fixing it

A quick demonstration of hidden state. Watch carefully.

In [ ]:
total = 10
total = total + 5
print("total is", total)

total is 15


### Step 2 --- Plan the package before writing it

Group code by **job**, not by the order you happened to write it.
Our project has three jobs, so it gets three modules.

```
work/
  delivery/
    __init__.py     marks this folder as a package
    data.py         load the CSV, split it
    features.py     check and describe a single order
    model.py        train, score, save, load
  train.py          the script a human runs
```

One rule: **a module should be describable in one sentence.** If you
cannot, it is doing two jobs and wants splitting.

In [ ]:
from pathlib import Path

WORK = Path("work")
PKG = WORK / "delivery"
PKG.mkdir(parents=True, exist_ok=True)

print("package folder ready:", PKG.resolve())

package folder ready: C:\SCSE3040-Lab\P04-package\work\delivery


### Step 3 --- Write the first module: data.py



In [ ]:
data_py = '''"""Loading and splitting the delivery dataset."""

from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split

FEATURES = ["distance_km", "prep_time_min", "traffic_level", "rain"]
TARGET = "delivery_min"
SEED = 42


def load_orders(path):
    """Read the delivery CSV into a table."""
    return pd.read_csv(Path(path))


def split_orders(orders, test_size=0.2):
    """Return X_train, X_test, y_train, y_test."""
    X = orders[FEATURES]
    y = orders[TARGET]
    return train_test_split(X, y, test_size=test_size,
                            random_state=SEED)
'''

(PKG / "data.py").write_text(data_py, encoding="utf-8")
print("wrote", PKG / "data.py", f"({len(data_py)} characters)")

wrote work\delivery\data.py (604 characters)


Notice what moved and what did not. The code is the same code from
P02. Only its address changed.

### Step 4 --- Write features.py

In [ ]:
features_py = '''"""Checks and descriptions for one delivery order."""

TRAFFIC_LEVELS = (1, 2, 3)


def minutes_per_km(delivery_min, distance_km):
    """How many minutes each kilometre took."""
    if distance_km <= 0:
        raise ValueError("distance_km must be positive")
    return delivery_min / distance_km


def describe_order(order):
    """A short sentence a human can read."""
    weather = "in the rain" if order["rain"] else "in dry weather"
    return (f"{order['distance_km']} km, "
            f"{order['prep_time_min']} min prep, "
            f"traffic {order['traffic_level']}, {weather}")
'''

(PKG / "features.py").write_text(features_py, encoding="utf-8")
print("wrote", PKG / "features.py")

wrote work\delivery\features.py


### Step 5 --- Write model.py


In [ ]:
model_py = '''"""Training, scoring, saving and loading the model."""

import joblib
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error


def train_model(X_train, y_train):
    """Fit a linear regression and hand it back."""
    return LinearRegression().fit(X_train, y_train)


def evaluate(model, X_test, y_test):
    """Mean absolute error, in minutes."""
    return mean_absolute_error(y_test, model.predict(X_test))


def save_model(model, path):
    """Write a trained model to disk."""
    joblib.dump(model, path)
    return path


def load_model(path):
    """Read a trained model back from disk."""
    return joblib.load(path)
'''

(PKG / "model.py").write_text(model_py, encoding="utf-8")
print("wrote", PKG / "model.py")

wrote work\delivery\model.py


### Step 6 --- Add __init__.py to make it a package


In [ ]:
init_py = '''"""Delivery-time prediction for SCSE3040."""

__version__ = "0.1.0"

from .data import FEATURES, TARGET, load_orders, split_orders
from .model import evaluate, load_model, save_model, train_model

__all__ = [
    "FEATURES", "TARGET", "load_orders", "split_orders",
    "train_model", "evaluate", "save_model", "load_model",
]
'''

(PKG / "__init__.py").write_text(init_py, encoding="utf-8")

for f in sorted(PKG.glob("*.py")):
    print(f"  {f.name:<15} {f.stat().st_size:>5} bytes")

  __init__.py       338 bytes
  data.py           627 bytes
  features.py       612 bytes
  model.py          696 bytes
  validate.py       363 bytes


### Step 7 --- Import your own package and use it


In [ ]:
import importlib
import shutil
import sys

if str(WORK.resolve()) not in sys.path:
    sys.path.insert(0, str(WORK.resolve()))


def fresh_import(name):
    """Import a module, reloading it if it changed on disk.

    The first line deletes Python's compiled cache. Python decides
    whether that cache is stale from the file's size and its
    timestamp to the nearest second -- so an edit that keeps the
    length the same can be missed entirely. Deleting it is simplest.
    """
    shutil.rmtree(PKG / "__pycache__", ignore_errors=True)
    importlib.invalidate_caches()
    module = importlib.import_module(name)
    return importlib.reload(module)


delivery_data = fresh_import("delivery.data")
delivery_model = fresh_import("delivery.model")

orders = delivery_data.load_orders(DATA)
X_train, X_test, y_train, y_test = delivery_data.split_orders(orders)
model = delivery_model.train_model(X_train, y_train)
mae = delivery_model.evaluate(model, X_test, y_test)

print(f"rows {len(orders)}  train {len(X_train)}  test {len(X_test)}")
print(f"MAE {mae:.2f} minutes")

rows 600  train 480  test 120
MAE 1.92 minutes


Four lines of your own library did what a screen of notebook cells
did in P02 --- and this time somebody else can run it.

### Step 8 --- Save the trained model to a file

Training takes time. Predicting should not repeat it. Save the
trained model once; load it whenever you need a prediction.

In [ ]:
model_path = WORK / "model.joblib"
delivery_model.save_model(model, model_path)

reloaded = delivery_model.load_model(model_path)
same = float(reloaded.predict(X_test.head(1))[0])
original = float(model.predict(X_test.head(1))[0])

print(f"saved to {model_path} "
      f"({model_path.stat().st_size:,} bytes)")
print(f"original model predicts {original:.2f}")
print(f"reloaded model predicts {same:.2f}")
print("identical:", abs(same - original) < 1e-9)

saved to work\model.joblib (968 bytes)
original model predicts 47.65
reloaded model predicts 47.65
identical: True


That file is the thing P07 will serve and P08 will put in a
container. From here on, "the model" means this file.

### Step 9 --- Write the script a human runs

Finally, the entry point: an ordinary Python file that a person, or
a scheduled job, or a container, can run with one command.

The `if __name__ == "__main__":` line means "only do this when
somebody runs this file directly, not when it is imported".

In [ ]:
train_py = '''"""Train the delivery-time model and report its error."""

import sys
from pathlib import Path

from delivery import (evaluate, load_orders, save_model,
                      split_orders, train_model)


HERE = Path(__file__).resolve().parent
DEFAULT_DATA = HERE.parent.parent / "data" / "delivery_times.csv"


def main():
    data_path = Path(sys.argv[1]) if len(sys.argv) > 1 else DEFAULT_DATA

    orders = load_orders(data_path)
    X_train, X_test, y_train, y_test = split_orders(orders)
    model = train_model(X_train, y_train)
    mae = evaluate(model, X_test, y_test)

    save_model(model, Path(__file__).parent / "model.joblib")

    print(f"rows: {len(orders)}")
    print(f"MAE: {mae:.2f}")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())
'''

(WORK / "train.py").write_text(train_py, encoding="utf-8")
print("wrote", WORK / "train.py")

wrote work\train.py


### Step 10 --- Run it as a real program

No notebook involved. This is the command you would put in a
Dockerfile in P08, or in a pipeline in P11.

In [ ]:
import subprocess

def run_script(name, args=(), cwd=WORK):
    """Run a python file the way a terminal would, and show output."""
    done = subprocess.run(
        [sys.executable, name, *args],
        cwd=cwd, capture_output=True, text=True,
    )
    print(f"$ python {name} {' '.join(args)}".rstrip())
    print(done.stdout.strip() or "(nothing printed)")
    if done.returncode != 0:
        print("STDERR:", done.stderr.strip()[-800:])
    print("exit code:", done.returncode)
    return done

result = run_script("train.py", [str(DATA.resolve())])

$ python train.py C:\SCSE3040-Lab\data\delivery_times.csv
rows: 600
MAE: 1.92
exit code: 0


Exit code `0` means success. Any other number means failure --- and
that is exactly how a pipeline in P11 will decide whether to carry
on or stop.

TASK-1

In [ ]:
new_function = '''

def average_speed_kmph(distance_km, delivery_min):
    "Average speed of a delivery, in kilometres per hour."
    return distance_km / (delivery_min / 60)
'''

with open(PKG / "features.py", "a", encoding="utf-8") as fh:
    fh.write(new_function)

feat = fresh_import("delivery.features")

T1_speed = feat.average_speed_kmph(10, 30)

print("10 km in 30 minutes =", T1_speed, "km/h")

10 km in 30 minutes = 20.0 km/h


TASK-2

In [ ]:
validate_py = '''"""Is this order usable?"""


def is_valid_order(order):
    """True if the order passes every rule."""

    if order["distance_km"] <= 0:
        return False

    if order["prep_time_min"] < 0:
        return False

    if order["traffic_level"] not in (1, 2, 3):
        return False

    if order["rain"] not in (0, 1):
        return False

    return True
'''

(PKG / "validate.py").write_text(validate_py, encoding="utf-8")

T2_validate = fresh_import("delivery.validate")

good = {
    "distance_km": 5.0,
    "prep_time_min": 20,
    "traffic_level": 2,
    "rain": 0
}

if T2_validate is None:
    print("Not done yet -- fill in the two TODOs above.")
else:
    print("good order ->", T2_validate.is_valid_order(good))

good order -> True


TASK-3

In [ ]:
predict_py = '''"""Predict one delivery, from the command line."""

import pandas as pd

from delivery import load_model


def main():
    model = load_model("model.joblib")

    order = pd.DataFrame([{
        "distance_km": 7,
        "prep_time_min": 25,
        "traffic_level": 3,
        "rain": 0
    }])

    minutes = float(model.predict(order)[0])

    print(f"PREDICTION: {minutes:.1f}")

    return 0


if __name__ == "__main__":
    raise SystemExit(main())
'''

(WORK / "predict.py").write_text(predict_py, encoding="utf-8")

T3_run = run_script("predict.py")

$ python predict.py
PREDICTION: 56.5
exit code: 0


---

## Self-check

Run the cell below to mark your work.

In [ ]:
# ------------------------------------------------------------------
# SELF-CHECK --- run this when you have attempted the tasks above.
# It never breaks your notebook. A task you have not done yet simply
# shows FAIL.
# ------------------------------------------------------------------

_results = []


def _check(label, fn):
    """Evaluate one graded condition without ever raising."""
    try:
        ok = bool(fn())
    except Exception:
        ok = False
    _results.append((label, ok))


_check('T1 | delivery.features now offers average_speed_kmph', lambda: hasattr(feat, 'average_speed_kmph'))
_check('T1 | 10 km in 30 minutes is 20 km/h', lambda: abs(float(T1_speed) - 20.0) < 1e-6)
_check('T1 | it still works for another order', lambda: abs(float(feat.average_speed_kmph(6, 45)) - 8.0) < 1e-6)
_check('T2 | delivery/validate.py exists and imports', lambda: (PKG / 'validate.py').is_file() and hasattr(T2_validate, 'is_valid_order'))
_check('T2 | a sensible order is accepted', lambda: T2_validate.is_valid_order({'distance_km': 5.0, 'prep_time_min': 20, 'traffic_level': 2, 'rain': 0}) is True)
_check('T2 | a zero distance is rejected', lambda: T2_validate.is_valid_order({'distance_km': 0, 'prep_time_min': 20, 'traffic_level': 2, 'rain': 0}) is False)
_check('T2 | traffic level 9 and rain 5 are both rejected', lambda: T2_validate.is_valid_order({'distance_km': 5.0, 'prep_time_min': 20, 'traffic_level': 9, 'rain': 0}) is False and T2_validate.is_valid_order({'distance_km': 5.0, 'prep_time_min': 20, 'traffic_level': 2, 'rain': 5}) is False)
_check('T3 | predict.py ran and exited successfully', lambda: T3_run.returncode == 0)
_check('T3 | it printed a PREDICTION line with a sensible number', lambda: 'PREDICTION:' in T3_run.stdout and 20 < float(T3_run.stdout.split('PREDICTION:')[1].split()[0]) < 90)

print("==================================================================")
print("SELF-CHECK   Practical 04 --- From Notebook to Package")
print("==================================================================")
for _label, _ok in _results:
    print(f"  [{'PASS' if _ok else 'FAIL'}]  {_label}")
print("------------------------------------------------------------------")
_passed = sum(1 for _, _ok in _results if _ok)
print(f"  {_passed} of {len(_results)} checks passed")
print("==================================================================")
if _passed == len(_results):
    print("Well done. Save the notebook and submit it.")
else:
    print("Read the FAIL lines above, fix those tasks, run this cell again.")

SELF-CHECK   Practical 04 --- From Notebook to Package
  [PASS]  T1 | delivery.features now offers average_speed_kmph
  [PASS]  T1 | 10 km in 30 minutes is 20 km/h
  [PASS]  T1 | it still works for another order
  [PASS]  T2 | delivery/validate.py exists and imports
  [PASS]  T2 | a sensible order is accepted
  [PASS]  T2 | a zero distance is rejected
  [PASS]  T2 | traffic level 9 and rain 5 are both rejected
  [PASS]  T3 | predict.py ran and exited successfully
  [PASS]  T3 | it printed a PREDICTION line with a sensible number
------------------------------------------------------------------
  9 of 9 checks passed
Well done. Save the notebook and submit it.
